# Transformer Baselines

This notebook runs Hugging Face transformer comparisons and creates analysis tables:

- `total_results`: all evaluated transformer runs.
- `report_results`: the best validation result per model.
- `epoch_history`: per-epoch validation metrics for training curves.

The task predicts sentiment labels `0..4`, with validation score `1 - MAE / 4`.

In [1]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

import pandas as pd
import torch

EXPERIMENT_KIND = "TRANSFORMER_BASELINES"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = Path("experiments/transformers") / f"{RUN_ID}_{EXPERIMENT_KIND}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("data/train.csv")
VALIDATION_SIZE = 0.1
RANDOM_STATE = 42
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
USE_FP16 = True
FP16_EXCLUDED_MODELS = {"microsoft/deberta-v3-base"}

EXPERIMENT_DIR

PosixPath('experiments/transformers/20260521_115409_TRANSFORMER_BASELINES')

In [2]:
models = [
    # {"model_name": "distilbert-base-uncased", "variant": "fine_tuned"},  # completed
    # {"model_name": "bert-base-uncased", "variant": "fine_tuned"},  # completed
    # {"model_name": "roberta-base", "variant": "fine_tuned"},  # completed
    {"model_name": "microsoft/deberta-v3-base", "variant": "fine_tuned"},
    {"model_name": "nlptown/bert-base-multilingual-uncased-sentiment", "variant": "fine_tuned"},
    {"model_name": "nlptown/bert-base-multilingual-uncased-sentiment", "variant": "eval_only"},
]

models

[{'model_name': 'microsoft/deberta-v3-base', 'variant': 'fine_tuned'},
 {'model_name': 'nlptown/bert-base-multilingual-uncased-sentiment',
  'variant': 'fine_tuned'},
 {'model_name': 'nlptown/bert-base-multilingual-uncased-sentiment',
  'variant': 'eval_only'}]

## Run Experiments

In [3]:
completed_runs = []

for spec in models:
    model_name = spec["model_name"]
    variant = spec["variant"]
    run_name = f"{model_name.replace('/', '__')}__{variant}"
    run_dir = EXPERIMENT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "-m",
        "baselines.train_review_model",
        "--model-name",
        model_name,
        "--experiment-name",
        f"transformer__{run_name}",
        "--train-path",
        str(TRAIN_PATH),
        "--output-dir",
        str(run_dir),
        "--validation-size",
        str(VALIDATION_SIZE),
        "--random-state",
        str(RANDOM_STATE),
        "--max-length",
        str(MAX_LENGTH),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
        "--learning-rate",
        str(LEARNING_RATE),
        "--weight-decay",
        str(WEIGHT_DECAY),
    ]
    if variant == "eval_only":
        cmd.append("--eval-only")
    if USE_FP16 and model_name not in FP16_EXCLUDED_MODELS:
        cmd.append("--fp16")

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    completed_runs.append({**spec, "run_name": run_name, "run_dir": run_dir})

completed_runs

/cluster/courses/cil/envs/envs/text-5060/bin/python -m baselines.train_review_model --model-name microsoft/deberta-v3-base --experiment-name transformer__microsoft__deberta-v3-base__fine_tuned --train-path data/train.csv --output-dir experiments/transformers/20260521_115409_TRANSFORMER_BASELINES/microsoft__deberta-v3-base__fine_tuned --validation-size 0.1 --random-state 42 --max-length 256 --epochs 3 --batch-size 16 --eval-batch-size 32 --learning-rate 2e-05 --weight-decay 0.01


Model: microsoft/deberta-v3-base
Variant: fine_tuned
Train examples: 226800
Validation examples: 25200
Class counts: {0: 45360, 1: 45360, 2: 45360, 3: 45360, 4: 45360}


Loading weights: 100%|████████████████████| 198/198 [00:00<00:00, 11949.41it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | 

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


...

 99%|███████████████████████████████████████▌| 780/788 [00:34<00:00, 24.62it/s]

100%|███████████████████████████████████████▉| 786/788 [00:35<00:00, 23.84it/s]

100%|████████████████████████████████████████| 788/788 [00:35<00:00, 22.45it/s]


Wrote experiments/transformers/20260521_115409_TRANSFORMER_BASELINES/nlptown__bert-base-multilingual-uncased-sentiment__eval_only/transformer_results.csv
Wrote experiments/transformers/20260521_115409_TRANSFORMER_BASELINES/nlptown__bert-base-multilingual-uncased-sentiment__eval_only/transformer_best_by_family.csv
Wrote experiments/transformers/20260521_115409_TRANSFORMER_BASELINES/nlptown__bert-base-multilingual-uncased-sentiment__eval_only/transformer_epoch_history.csv
[ok] transformer__nlptown__bert-base-multilingual-uncased-sentiment__eval_only: best_epoch=0.0 score=0.89336 mae=0.42655 acc=0.63401 macro_f1=0.62847


[{'model_name': 'microsoft/deberta-v3-base',
  'variant': 'fine_tuned',
  'run_name': 'microsoft__deberta-v3-base__fine_tuned',
  'run_dir': PosixPath('experiments/transformers/20260521_115409_TRANSFORMER_BASELINES/microsoft__deberta-v3-base__fine_tuned')},
 {'model_name': 'nlptown/bert-base-multilingual-uncased-sentiment',
  'variant': 'fine_tuned',
  'run_name': 'nlptown__bert-base-multilingual-uncased-sentiment__fine_tuned',
  'run_dir': PosixPath('experiments/transformers/20260521_115409_TRANSFORMER_BASELINES/nlptown__bert-base-multilingual-uncased-sentiment__fine_tuned')},
 {'model_name': 'nlptown/bert-base-multilingual-uncased-sentiment',
  'variant': 'eval_only',
  'run_name': 'nlptown__bert-base-multilingual-uncased-sentiment__eval_only',
  'run_dir': PosixPath('experiments/transformers/20260521_115409_TRANSFORMER_BASELINES/nlptown__bert-base-multilingual-uncased-sentiment__eval_only')}]

## Total Analysis

In [4]:
result_frames = []
for run in completed_runs:
    result_frames.append(pd.read_csv(run["run_dir"] / "transformer_results.csv"))

results = pd.concat(result_frames, ignore_index=True)
total_results = results.sort_values(
    ["status", "cil_score"], ascending=[False, False]
).reset_index(drop=True)
total_results.to_csv(EXPERIMENT_DIR / "total_analysis.csv", index=False)
total_results

,experiment,family,model,variant,status,best_epoch,best_epoch_train_loss,best_epoch_val_loss,train_seconds,eval_seconds,accuracy,macro_f1,mae,cil_score,quadratic_weighted_kappa,notes
0,transformer__nlptown__bert-base-multilingual-u...,transformer,nlptown/bert-base-multilingual-uncased-sentiment,fine_tuned,ok,3.0,0.680230,0.850999,3884.967314,35.298712,0.644683,0.643910,0.403254,0.899187,0.871509,Final eval score=0.89919
1,transformer__nlptown__bert-base-multilingual-u...,transformer,nlptown/bert-base-multilingual-uncased-sentiment,eval_only,ok,0.0,NaN,0.836051,0.000000,35.693490,0.634008,0.628472,0.426548,0.893363,0.863253,Final eval score=0.89336
2,transformer__microsoft__deberta-v3-base__fine_...,transformer,microsoft/deberta-v3-base,fine_tuned,ok,1.0,1.614795,NaN,4930.337912,63.419465,0.200000,0.066667,2.000000,0.500000,0.000000,Final eval score=0.50000


## Report Analysis

In [5]:
ok_results = results[results["status"] == "ok"].copy()
report_results = (
    ok_results.sort_values("cil_score", ascending=False)
    .groupby(["model", "variant"], as_index=False)
    .first()
    .sort_values("cil_score", ascending=False)
    .reset_index(drop=True)
)
report_results.to_csv(EXPERIMENT_DIR / "report_analysis.csv", index=False)
report_results

,model,variant,experiment,family,status,best_epoch,best_epoch_train_loss,best_epoch_val_loss,train_seconds,eval_seconds,accuracy,macro_f1,mae,cil_score,quadratic_weighted_kappa,notes
0,nlptown/bert-base-multilingual-uncased-sentiment,fine_tuned,transformer__nlptown__bert-base-multilingual-u...,transformer,ok,3.0,0.680230,0.850999,3884.967314,35.298712,0.644683,0.643910,0.403254,0.899187,0.871509,Final eval score=0.89919
1,nlptown/bert-base-multilingual-uncased-sentiment,eval_only,transformer__nlptown__bert-base-multilingual-u...,transformer,ok,0.0,NaN,0.836051,0.000000,35.693490,0.634008,0.628472,0.426548,0.893363,0.863253,Final eval score=0.89336
2,microsoft/deberta-v3-base,fine_tuned,transformer__microsoft__deberta-v3-base__fine_...,transformer,ok,1.0,1.614795,NaN,4930.337912,63.419465,0.200000,0.066667,2.000000,0.500000,0.000000,Final eval score=0.50000


## Epoch History

In [6]:
history_frames = []
for run in completed_runs:
    history_frames.append(pd.read_csv(run["run_dir"] / "transformer_epoch_history.csv"))

epoch_history = pd.concat(history_frames, ignore_index=True)
epoch_history = epoch_history.sort_values(["experiment", "epoch"]).reset_index(drop=True)
epoch_history.to_csv(EXPERIMENT_DIR / "epoch_analysis.csv", index=False)
epoch_history

,experiment,epoch,train_loss,val_loss,accuracy,macro_f1,mae,cil_score,quadratic_weighted_kappa
0,transformer__microsoft__deberta-v3-base__fine_...,1.0,1.614795,NaN,0.200000,0.066667,2.000000,0.500000,0.000000
1,transformer__microsoft__deberta-v3-base__fine_...,2.0,0.000000,NaN,0.200000,0.066667,2.000000,0.500000,0.000000
2,transformer__microsoft__deberta-v3-base__fine_...,3.0,0.000000,NaN,0.200000,0.066667,2.000000,0.500000,0.000000
3,transformer__nlptown__bert-base-multilingual-u...,0.0,NaN,0.836051,0.634008,0.628472,0.426548,0.893363,0.863253
4,transformer__nlptown__bert-base-multilingual-u...,1.0,0.858159,0.834058,0.636905,0.639880,0.408333,0.897917,0.866883
5,transformer__nlptown__bert-base-multilingual-u...,2.0,0.772469,0.813666,0.646746,0.644559,0.403373,0.899157,0.871656
6,transformer__nlptown__bert-base-multilingual-u...,3.0,0.680230,0.850999,0.644683,0.643910,0.403254,0.899187,0.871509
